### Optional: consider removing cloud-masked training samples, especially from the positive training dataset

A few partly-masked images in the negative dataset is probably healthy.

Expected patch layout: `WORK_DIR / <source_stem> / train / {0|1} / *.tif`.

In [ ]:
# >> FILE OVERVIEW: training-data cleaning notebook (upstream Amazon Mining Watch).
# >> NOT part of production scoring: it runs once, by hand, before model training.
# >> Goal: find training patches with "black" (cloud-masked, value 0) pixels and move
# >> the positive ones out of the training set, so the CNN never learns "mine = hole".
# >> Input layout: WORK_DIR/<source>/train/{0|1}/*.tif  (0 = no mine, 1 = mine).
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

# >> Add the repo root to Python's import path so `deprecated.viz_tools` can be found.
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# >> GOTCHA: `deprecated/` (with viz_tools.plot_numpy_grid) is archived in this fork,
# >> so this import fails and the notebook will not run as-is.
from deprecated import viz_tools

# >> Folder of exported training patches; the timestamp names one export run.
WORK_DIR = Path("../data/training_patches2026-03-24T23:15/")

%load_ext autoreload
%autoreload 2


In [ ]:
# >> Read one GeoTIFF patch into memory.
# >> rasterio gives (bands, rows, cols); moveaxis makes it (rows, cols, bands),
# >> i.e. (13, 48, 48) -> (48, 48, 13).
# >> Values stay raw reflectance x 10,000 (no /10000 here), cast to float32.
def read_tif(path):
    with rasterio.open(path) as f:
        img = f.read()
    img = np.moveaxis(img, 0, -1)
    return img.astype('float32')

# >> Positives = label folder 1, negatives = folder 0. sorted() keeps order stable,
# >> so the Nth patch array lines up with the Nth path (used later to move files).
# Layout: WORK_DIR / <source_stem> / train / {0|1} / *.tif
positive_paths = sorted(WORK_DIR.glob("*/train/1/*.tif"))
# >> Stack every patch into one array: shape (N, 48, 48, 13), float32.
positive_patches = np.array([read_tif(p) for p in positive_paths])
negative_paths = sorted(WORK_DIR.glob("*/train/0/*.tif"))
negative_patches = np.array([read_tif(p) for p in negative_paths])

print(len(positive_patches), "positive samples loaded")
print(len(negative_patches), "negative samples loaded")


In [ ]:
# >> Measure how much of each patch is "black" (masked) and split pass / reject.
# >> Uses only band index 1 = B2 (blue, 490 nm); slice 1:2 keeps a length-1 band axis,
# >> so the mean over axis -1 is just B2 itself. Shape per patch: (48, 48).
# >> dark_level=10 is in raw units (reflectance x 10,000), i.e. reflectance < 0.001.
# >> masked_fraction = share of pixels darker than that (0.0 - 1.0).
# >> Returns (kept patches, rejected patches, indices of rejected patches in `data`).
def filter_black(data, mask_limit=0.1, dark_level=10):
    masked_fraction = np.array([np.sum(np.mean(patch[:,:,1:2], axis=-1) < dark_level) / np.size(np.mean(patch[:,:,1:2], axis=-1)) for patch in data])
    # >> Keep patches whose dark fraction is at or below mask_limit.
    filtered_data = data[masked_fraction <= mask_limit]
    print(f"{len(filtered_data) / len(data) :.4%} of data passes filter")

    # >> Indices of rejects; used later to find the matching file paths.
    rejected_indices = np.where(masked_fraction > mask_limit)[0]
    rejected_data = data[rejected_indices]
    return filtered_data, rejected_data, rejected_indices


In [ ]:
# Filter pixels that are masked beyond a threshold. We want to avoid giving positive examples of cloud-masked patches
# >> mask_limit=0: a single dark pixel is enough to reject a positive patch.
filtered, rejects, rejected_indices = filter_black(positive_patches, mask_limit = 0)

if len(rejects) > 0:
    num_samples = min(len(rejects), 256)
    # >> RGB preview: slice 3:0:-1 = indices 3,2,1 = B4 red, B3 green, B2 blue.
    # >> /3000 scales reflectance x 10,000 to roughly 0-1 for display.
    rgbs = rejects[:,:,:,3:0:-1] / 3000
    # >> Show up to 256 random rejects as a grid (needs the archived viz_tools).
    viz_samples = random.sample(list(rgbs), num_samples)
    viz_tools.plot_numpy_grid(viz_samples)
    plt.title("Masked Rejects")

In [ ]:
# >> FILE MOVE (not a copy): each rejected POSITIVE .tif goes to a sibling folder
# >> train/1/cloud_mask_rejects/, which the train/1/*.tif glob no longer sees.
# >> Relies on rejected_indices from the positive run in the cell above.
for src in np.array(positive_paths, dtype=object)[rejected_indices]:
    src = Path(src)
    dest_dir = src.parent / "cloud_mask_rejects"
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / src.name
    shutil.move(src, dest)


In [ ]:
# Filter pixels that are masked beyond a threshold. 
# >> Same check on NEGATIVES. This overwrites `filtered`, `rejects`, `rejected_indices`,
# >> so cells below now refer to the negative set.
filtered, rejects, rejected_indices = filter_black(negative_patches, mask_limit = 0)

if len(rejects) > 0:
    num_samples = min(len(rejects), 256)
    rgbs = rejects[:,:,:,3:0:-1] / 3000
    viz_samples = random.sample(list(rgbs), num_samples)
    viz_tools.plot_numpy_grid(viz_samples)
    plt.title("Masked Rejects")

In [ ]:
num_samples = 16 ** 2
# >> Show 256 random negatives that passed (random with replacement, so repeats occur).
indices = np.random.randint(0, len(filtered), num_samples)
fig = viz_tools.plot_numpy_grid(filtered[indices,:,:,3:0:-1] / 3000)
plt.title('Filtered Samples')
plt.show()

In [ ]:
# >> Paths of the negative rejects, shuffled so a subset would be a random sample.
cloudy = np.array(negative_paths)[rejected_indices]
random.shuffle(cloudy)
print(len(cloudy))

In [ ]:
# >> n_to_remove = 0: no negatives are moved; they are only displayed above.
# >> Some partly-masked negatives are kept on purpose (see the markdown at the top).
# >> Raise this number to move that many random negative rejects out of train/0/.
n_to_remove = 0
for src in cloudy[:n_to_remove]:
    src = Path(src)
    dest_dir = src.parent / "cloud_mask_rejects"
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / src.name
    shutil.move(src, dest)
